In [26]:
import pandas as pd
import numpy as np
import os
import json
import openpyxl
import subprocess
from collections import Counter
from sklearn.model_selection import GroupShuffleSplit

# Paths
GUIDE_SEQ_PATH = 'SysEvalOffTarget/files/datasets/GUIDE-seq.xlsx'
CHANGE_SEQ_PATH = 'SysEvalOffTarget/files/datasets/CHANGE-seq.xlsx'

## Load and Inspect Raw Datasets

In [12]:
guide_seq = pd.read_excel(GUIDE_SEQ_PATH, engine='openpyxl')
change_seq = pd.read_excel(CHANGE_SEQ_PATH, engine='openpyxl')

for name, df in [('GUIDE-seq', guide_seq), ('CHANGE-seq', change_seq)]:
    print(f'--- {name} ---')
    print(f'Shape: {df.shape}')
    print(f'Columns: {list(df.columns)}')
    print(f'Dtypes:\n{df.dtypes}')
    print(f'Missing values:\n{df.isnull().sum()}')
    print()

--- GUIDE-seq ---
Shape: (1702, 11)
Columns: ['chrom', 'chromStart', 'chromEnd', 'name', 'GUIDEseq_reads', 'strand', 'offtarget_sequence', 'genomic_coordinate', 'distance', 'target', 'run']
Dtypes:
chrom                 object
chromStart             int64
chromEnd               int64
name                  object
GUIDEseq_reads         int64
strand                object
offtarget_sequence    object
genomic_coordinate    object
distance               int64
target                object
run                    int64
dtype: object
Missing values:
chrom                 0
chromStart            0
chromEnd              0
name                  0
GUIDEseq_reads        0
strand                0
offtarget_sequence    0
genomic_coordinate    0
distance              0
target                0
run                   0
dtype: int64

--- CHANGE-seq ---
Shape: (202043, 11)
Columns: ['chrom', 'chromStart', 'chromEnd', 'name', 'CHANGEseq_reads', 'strand', 'offtarget_sequence', 'Unnamed: 7', 'chromStart:chromE

In [19]:
guide_seq.head()

,chrom,chromStart,chromEnd,name,GUIDEseq_reads,strand,offtarget_sequence,genomic_coordinate,distance,target,run,label
0,chr19,55115744,55115767,AAVS1_site_1,13557,+,GTCACCAATCCTGTCCCTAGTGG,chr19:55115745-55115767:+,0,GTCACCAATCCTGTCCCTAGNGG,1,1
1,chrX,1450701,1450724,AAVS1_site_1,190,+,CTCCCCAACCCCATCCCTAGGGG,chrX:1450702-1450724:+,5,GTCACCAATCCTGTCCCTAGNGG,1,1
2,chrX,1452125,1452148,AAVS1_site_1,105,+,CTCCCCAACCCCATCCCTAGGGG,chrX:1452126-1452148:+,5,GTCACCAATCCTGTCCCTAGNGG,1,1
3,chr1,12523844,12523867,AAVS1_site_1,2,+,CACACTAATCCTGTCCCCAGAGG,chr1:12523845-12523867:+,4,GTCACCAATCCTGTCCCTAGNGG,1,1
4,chr19,55115744,55115767,AAVS1_site_1,55079,+,GTCACCAATCCTGTCCCTAGTGG,chr19:55115745-55115767:+,0,GTCACCAATCCTGTCCCTAGNGG,2,1


In [20]:
change_seq.head()

,chrom,chromStart,chromEnd,name,CHANGEseq_reads,strand,offtarget_sequence,Unnamed: 7,chromStart:chromEnd,distance,target,label
0,chr4,121343302,121343325,AAVS1_site_1,540,+,ATCACCTATCCTATCCCTAAGGG,NaN,chr4:121343303-121343325:+,4,GTCACCAATCCTGTCCCTAGNGG,1
1,chr1,12523844,12523867,AAVS1_site_1,314,+,CACACTAATCCTGTCCCCAGAGG,NaN,chr1:12523845-12523867:+,4,GTCACCAATCCTGTCCCTAGNGG,1
2,chr8,66370990,66371013,AAVS1_site_1,258,-,AGCATAAATCCTGTCCCTAGGAG,NaN,chr8:66370991-66371013:-,5,GTCACCAATCCTGTCCCTAGNGG,1
3,chr9,134937838,134937861,AAVS1_site_1,226,-,AAAACCAAACCTGTCCCTAAAGG,NaN,chr9:134937839-134937861:-,5,GTCACCAATCCTGTCCCTAGNGG,1
4,chr15,36995651,36995674,AAVS1_site_1,130,+,TGATCCTATCCTGTCCCTAGAGG,NaN,chr15:36995652-36995674:+,5,GTCACCAATCCTGTCCCTAGNGG,1


In [14]:
GRNA_COL = 'target'
TARGET_COL = 'offtarget_sequence'
GUIDE_LABEL_COL = 'GUIDEseq_reads'
CHANGE_LABEL_COL = 'CHANGEseq_reads'

# Checking sequence lengths before filtering
print('GUIDE-seq target lengths:')
print(guide_seq[GRNA_COL].str.len().value_counts())
print('\nGUIDE-seq offtarget lengths:')
print(guide_seq[TARGET_COL].str.len().value_counts())

print('\nCHANGE-seq target lengths:')
print(change_seq[GRNA_COL].str.len().value_counts())
print('\nCHANGE-seq offtarget lengths:')
print(change_seq[TARGET_COL].str.len().value_counts())

GUIDE-seq target lengths:
target
23    1702
Name: count, dtype: int64

GUIDE-seq offtarget lengths:
offtarget_sequence
23    1702
Name: count, dtype: int64

CHANGE-seq target lengths:
target
23    202043
Name: count, dtype: int64

CHANGE-seq offtarget lengths:
offtarget_sequence
23.0    198881
24.0      3114
Name: count, dtype: int64


## Parse and Align Sequences

Each entry should have:
- A 20 nucleotide gRNA sequence
- A 23 nucleotide genomic target sequence (20nt + 3nt PAM)

In [15]:
def parse_sequences(df, grna_col, target_col):
    df = df.copy()

    # Drop missing rows
    df = df.dropna(subset=[grna_col, target_col])

    df[grna_col] = df[grna_col].str.upper().str.strip()
    df[target_col] = df[target_col].str.upper().str.strip()

    # Both sequences are 23nt (20nt + 3nt PAM)
    valid = (df[grna_col].str.len() == 23) & (df[target_col].str.len() == 23)
    n_dropped = (~valid).sum()
    if n_dropped > 0:
        print(f'Dropping {n_dropped} rows with unexpected sequence lengths')
    return df[valid].reset_index(drop=True)

guide_seq = parse_sequences(guide_seq, GRNA_COL, TARGET_COL)
change_seq = parse_sequences(change_seq, GRNA_COL, TARGET_COL)

print(f'GUIDE-seq after parsing: {guide_seq.shape}')
print(f'CHANGE-seq after parsing: {change_seq.shape}')

Dropping 3114 rows with unexpected sequence lengths
GUIDE-seq after parsing: (1702, 11)
CHANGE-seq after parsing: (198881, 11)


## Label On-Target vs Off-Target

In GUIDE-seq and CHANGE-seq, entries with a read count above zero are off-target cleavage sites. On-target entries (gRNA perfectly matching the intended site) are labelled 1; off-target labelled 0.

In [16]:
def assign_labels(df, label_col):
    """
    1 = off-target cleavage detected (reads > 0)
    0 = no cleavage
    """
    df = df.copy()
    df['label'] = (df[label_col] > 0).astype(int)
    return df

guide_seq = assign_labels(guide_seq, GUIDE_LABEL_COL)
change_seq = assign_labels(change_seq, CHANGE_LABEL_COL)

for name, df in [('GUIDE-seq', guide_seq), ('CHANGE-seq', change_seq)]:
    counts = df['label'].value_counts()
    pct = df['label'].value_counts(normalize=True) * 100
    print(f'{name} label distribution:')
    print(pd.DataFrame({'count': counts, 'percent': pct.round(1)}))
    print()

GUIDE-seq label distribution:
       count  percent
label                
1       1702    100.0

CHANGE-seq label distribution:
        count  percent
label                 
1      198881    100.0



### One-time Data Preparation
Run this cell once to generate the positive and negative datasets.
Took me ~10-15 minutes to run

In [29]:
result = subprocess.run(['python', 'SysEvalOffTarget/prepare_data.py'], 
                      capture_output=True, text=True)
print(result.stdout)
print(result.stderr)

create CHANGE-seq dataset
number of optional off targets before filtering:  3567646
Dropping chroms which do not appear in the experiment
number of optional off targets after this stage:  3385057
Dropping off-targets which their target doesn't appear in the experiment_df
number of optional off targets after this stage:  3385057
Dropping for each Target the optional off-targets which their sequences         (or their reverse complement) appear in the experiment (without connection to chromStart)
number of optional off targets after this stage:  2826927
Dropping for each chrom the optional off-targets which their chromStart appear in the experiment
number of optional off targets after this stage:  2821758
Dropping for each Target duplicates of optional off-targets
number of optional off targets after this stage:  2806152
dropping on-targets if exists (at all and after all the previous stages(
number of optional off targets after this stage:  2806151
create GUIDE-seq dataset
number of opt